# 01 - 投影转换与面积计算

**目标**：读取 haidian 临时边界 GeoJSON (EPSG:4326)，转换到 EPSG:4548，计算面积，与官方声明值对比。

**学习要点**：
- shapely 几何对象 → pyproj 投影变换 → 面积计算
- EPSG:4548 是中国城市设计常用投影坐标系（CGCS2000 / 3-degree Gauss-Kruger CM 117E，单位：米）
- 理解为什么不能在 EPSG:4326（经纬度）下直接算面积

In [1]:
import sys
sys.path.insert(0, '..')

import geopandas as gpd
import pyproj
from shapely.ops import transform
from shapely.geometry import Polygon, shape
import json

from src.projection import (
    transform_geometry,
    compute_area_4548,
    format_area,
    CRS_4326,
    CRS_4548,
)

print('Libraries loaded successfully.')

Libraries loaded successfully.


In [2]:
# Load the haidian boundary GeoJSON
BOUNDARY_PATH = '../data/haidian-boundary.geojson'

gdf_all = gpd.read_file(BOUNDARY_PATH)
print(f'Loaded {len(gdf_all)} features')
print(f'CRS: {gdf_all.crs}')
print(f'Columns: {list(gdf_all.columns)}')
print()

# Inspect each feature
for _, row in gdf_all.iterrows():
    geom_type = row.geometry.geom_type
    name = row.get('name_zh', row.get('id', '?'))
    declared = row.get('area_sqm_declared', 'N/A')
    calculated = row.get('area_sqm_calculated', 'N/A')
    print(f'  {row["id"]}: {name} ({geom_type})')
    print(f'    Declared: {declared} sqm, Pre-calculated: {calculated} sqm')
    print()

Loaded 6 features
CRS: EPSG:4326
Columns: ['id', 'layer', 'parent_scope_id', 'area_id', 'scope_id', 'name_zh', 'source_type', 'confidence', 'geometry_role', 'official_boundary', 'boundary_precision', 'announced_area_sqm', 'source_id', 'source_title', 'area_sqm_declared', 'area_sqm_calculated', 'usage_note', 'geometry']

  PROV-SITE-001: 总体设计范围粗略替代边界 (Polygon)
    Declared: 11400000.0 sqm, Pre-calculated: 11412825.386 sqm

  PROV-RESEARCH-001: 统筹研究范围粗略替代边界 (Polygon)
    Declared: 43600000.0 sqm, Pre-calculated: 43609232.558 sqm

  PROV-KEY-SCOPE-001: 重点区域范围粗略替代边界 (MultiPolygon)
    Declared: 3684000.0 sqm, Pre-calculated: 3692893.005 sqm

  PROV-KEY-001: 众智园AI自主创新加速区粗略范围 (Polygon)
    Declared: nan sqm, Pre-calculated: 1929201.877 sqm

  PROV-KEY-002: 北京AI原点社区粗略范围 (Polygon)
    Declared: nan sqm, Pre-calculated: 1043236.909 sqm

  PROV-KEY-003: 大钟寺AI产业聚集区粗略范围 (Polygon)
    Declared: nan sqm, Pre-calculated: 720454.221 sqm



## 选择总体设计范围进行验证

从 `provisional_boundaries.geojson` 中选取 `scope_id = 'overall_design_area'` 的 feature (PROV-SITE-001)。
声明面积为 11,400,000 m²（约 11.4 km²）。

In [3]:
# Extract the overall design area boundary
site_feature = gdf_all[gdf_all['id'] == 'PROV-SITE-001'].iloc[0]
boundary_4326 = site_feature.geometry
declared_area = site_feature['area_sqm_declared']  # 11,400,000 sqm

print(f'Feature: {site_feature["name_zh"]}')
print(f'Geometry type: {boundary_4326.geom_type}')
print(f'CRS: EPSG:4326 (WGS84 lat/lng)')
print(f'Declared area: {declared_area:,} sqm = {declared_area/1e6:.2f} km²')
print()
print(f'Bounding box (lon/lat): {boundary_4326.bounds}')

Feature: 总体设计范围粗略替代边界
Geometry type: Polygon
CRS: EPSG:4326 (WGS84 lat/lng)
Declared area: 11,400,000.0 sqm = 11.40 km²

Bounding box (lon/lat): (116.3397, 39.939, 116.3553, 40.0265)


## 坐标转换：EPSG:4326 → EPSG:4548

使用 `pyproj.Transformer` 进行坐标转换。EPSG:4548 是 CGCS2000 / 3-degree Gauss-Kruger CM 117E 坐标系，单位是米。

In [4]:
# Create transformer
transformer = pyproj.Transformer.from_crs(CRS_4326, CRS_4548, always_xy=True)

# Transform the geometry
boundary_4548 = transform(transformer.transform, boundary_4326)

print(f'Original CRS: EPSG:4326')
print(f'  Bounds (lon/lat): ({boundary_4326.bounds[0]:.4f}, {boundary_4326.bounds[1]:.4f}) to ({boundary_4326.bounds[2]:.4f}, {boundary_4326.bounds[3]:.4f})')
print()
print(f'Transformed CRS: EPSG:4548')
print(f'  Bounds (metres): ({boundary_4548.bounds[0]:.0f}, {boundary_4548.bounds[1]:.0f}) to ({boundary_4548.bounds[2]:.0f}, {boundary_4548.bounds[3]:.0f})')
print()

# Verify: also try the reusable function
boundary_4548_v2 = transform_geometry(boundary_4326, CRS_4326, CRS_4548)
print(f'Reusable function gives same result: {boundary_4548.equals(boundary_4548_v2)}')

Original CRS: EPSG:4326
  Bounds (lon/lat): (116.3397, 39.9390) to (116.3553, 40.0265)

Transformed CRS: EPSG:4548
  Bounds (metres): (443594, 4422955) to (444968, 4432678)

Reusable function gives same result: True


## 面积计算

在 EPSG:4548 下使用 `shapely.geometry.Polygon.area` 计算面积，单位是平方米。

In [5]:
# Calculate area in EPSG:4548
area_sqm = boundary_4548.area
area_ha = area_sqm / 10_000
area_km2 = area_sqm / 1_000_000

print('=== Area Calculation ===')
print(f'Area: {area_sqm:,.2f} sqm')
print(f'     : {area_ha:,.2f} ha')
print(f'     : {area_km2:,.4f} km²')
print()

# Compare with declared area
deviation_sqm = area_sqm - declared_area
deviation_pct = (deviation_sqm / declared_area) * 100

print(f'=== Comparison with Declared Value ===')
print(f'Declared area:  {declared_area:,.2f} sqm ({declared_area/1e6:.4f} km²)')
print(f'Calculated:     {area_sqm:,.2f} sqm ({area_km2:.4f} km²)')
print(f'Deviation:      {deviation_sqm:+,.2f} sqm ({deviation_pct:+.4f}%)')
print()

if abs(deviation_pct) <= 5:
    print(f'✓ Deviation within ±5% tolerance — acceptable for provisional boundary')
else:
    print(f'✗ Deviation exceeds ±5% — needs investigation')

=== Area Calculation ===
Area: 11,412,825.39 sqm
     : 1,141.28 ha
     : 11.4128 km²

=== Comparison with Declared Value ===
Declared area:  11,400,000.00 sqm (11.4000 km²)
Calculated:     11,412,825.39 sqm (11.4128 km²)
Deviation:      +12,825.39 sqm (+0.1125%)

✓ Deviation within ±5% tolerance — acceptable for provisional boundary


In [6]:
# Also test with the reusable function (working on GeoDataFrame)
gdf_site = gpd.GeoDataFrame(
    [{'name': 'overall_design_area', 'geometry': boundary_4326}],
    crs=CRS_4326
)
gdf_with_area = compute_area_4548(gdf_site)
print(f'Area via compute_area_4548: {gdf_with_area["area_sqm"].iloc[0]:,.2f} sqm')
print()

# Format area
formatted = format_area(area_sqm)
print(f'Formatted: {formatted}')

Area via compute_area_4548: 11,412,825.39 sqm

Formatted: {'sqm': 11412825.39, 'ha': 1141.2825, 'km2': 11.412825}


## 验证：多边界面积计算

对所有三个范围内的边界都进行面积计算和对比。

In [7]:
# Select site boundary features
site_features = gdf_all[gdf_all['layer'] == 'SITE_BOUNDARY'].copy()

print('=== Multi-boundary Area Comparison ===')
print()

for _, row in site_features.iterrows():
    geom = row.geometry
    geom_4548 = transform_geometry(geom, CRS_4326, CRS_4548)
    calc_area = geom_4548.area
    declared = row['area_sqm_declared']
    deviation = (calc_area - declared) / declared * 100
    
    print(f'{row["id"]}: {row["name_zh"]}')
    print(f'  Declared:  {declared:,.0f} sqm = {declared/1e6:.4f} km²')
    print(f'  Calculated: {calc_area:,.0f} sqm = {calc_area/1e6:.4f} km²')
    print(f'  Deviation: {deviation:+.4f}%')
    print()

=== Multi-boundary Area Comparison ===



PROV-SITE-001: 总体设计范围粗略替代边界
  Declared:  11,400,000 sqm = 11.4000 km²
  Calculated: 11,412,825 sqm = 11.4128 km²
  Deviation: +0.1125%

PROV-RESEARCH-001: 统筹研究范围粗略替代边界
  Declared:  43,600,000 sqm = 43.6000 km²
  Calculated: 43,609,233 sqm = 43.6092 km²
  Deviation: +0.0212%



PROV-KEY-SCOPE-001: 重点区域范围粗略替代边界
  Declared:  3,684,000 sqm = 3.6840 km²
  Calculated: 3,692,893 sqm = 3.6929 km²
  Deviation: +0.2414%



## 为什么不能在 EPSG:4326 下直接算面积？

EPSG:4326 使用经纬度（度），不是平面坐标。直接用 `shapely.area` 算出来的"面积"单位是平方度，没有物理意义。

此外，在高纬度地区，1 度经度对应的实际距离远小于赤道处 — 所以必须使用等面积/等距投影坐标系（如 EPSG:4548）来进行精确的面积测量。

In [8]:
# Demonstrate: area in EPSG:4326 (wrong!)
area_in_degrees = boundary_4326.area
print(f'Area in EPSG:4326 (degrees²): {area_in_degrees:.10f}')
print(f'This value is in square degrees — NOT square metres!')
print(f'It is meaningless for real-world measurement.')

Area in EPSG:4326 (degrees²): 0.0012032500
This value is in square degrees — NOT square metres!
It is meaningless for real-world measurement.


## 总结

- 成功将 haidian 临时边界从 EPSG:4326 转换到 EPSG:4548
- 计算面积与声明值对比，验证偏差在可接受范围内
- `src/projection.py` 提供了可复用的投影和面积计算函数
- 下一步：生成模拟的用地分区数据